# Response plot -- reliable & SMI-valid cells (JSY090, Day5)

Peak-sorted, split-half response plot (`helper/ResponseVisualization.py`'s `create_response_plot`, reused **unchanged** -- existing pipeline code) for the cells in JSY090's Day5 (baseline) session that pass BOTH criteria:

- **reliability test** -- `analysis_reliable_cells` (`combined_reliable` & non-onset, saved in Phase 3's `*_smi_results_dreadd.h5` under `global_smi/analysis_reliable_cells`)
- **SMI validity** -- `valid_cells_mask` (`reliable_valid_cells`: reliable AND the SMI curve fit succeeded, same `global_smi/valid_cells_mask`)

If you actually meant the raw `combined_reliable` (from `preproc.h5`, before the onset filter) rather than `analysis_reliable_cells` for "reliability test," that's a one-line swap below -- flagged where it happens.

`create_response_plot` needs `norm_spatial_activity`, which lives in `preproc.h5` (not Phase 3's saved SMI output, whose own `bin_centers` is rescaled for curve fitting -- same caveat noted throughout this project's other notebooks).

In [ ]:
import sys
sys.path.insert(0, r"C:\Users\jasmineyeo\Documents\GitHub\V1_SpatialModulation")

import os
import re
import glob

import numpy as np
import h5py
import matplotlib.pyplot as plt
from matplotlib import rcParams

rcParams['legend.fontsize'] = 20
rcParams['axes.labelsize'] = 20
rcParams['axes.titlesize'] = 25
rcParams['xtick.labelsize'] = 20
rcParams['ytick.labelsize'] = 20

# Existing pipeline code, reused via import -- not modified.
from helper import files
from helper.ResponseVisualization import create_response_plot

ANIMAL_DIR = r"D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD"

## Locate JSY090's Day5 session

Same `*_smi_results_dreadd.h5` discovery + `Day<N>` label convention as `4.SessionComparison.py`'s `discover_smi_sessions` (Function 4.2), scoped down to just finding Day5 rather than cataloging every session.

In [ ]:
def find_day5_session(animal_dir, day_number=5):
    """
    Locate one baseline day's TSeries folder for animal_dir, using the
    same 'Day<N>' parent-folder convention discover_smi_sessions/
    discover_animal_sessions use elsewhere in this project.

    Parameters
    ----------
    animal_dir : str
    day_number : int

    Returns
    -------
    tseries_dir : str
    """
    save_paths = sorted(glob.glob(os.path.join(animal_dir, '**', '*_smi_results_dreadd.h5'), recursive=True))

    matches = []
    for save_path in save_paths:
        tseries_dir = os.path.dirname(save_path)
        parent_name = os.path.basename(os.path.dirname(tseries_dir))
        day_match = re.search(r'Day(\d+)', parent_name, re.IGNORECASE)
        if day_match and int(day_match.group(1)) == day_number:
            matches.append(tseries_dir)

    if not matches:
        raise FileNotFoundError(f"No Day{day_number} session with a saved *_smi_results_dreadd.h5 "
                                 f"found under {animal_dir}.")
    if len(matches) > 1:
        print(f"WARNING: {len(matches)} Day{day_number} matches, using the first: {matches}")

    print(f"Day{day_number} -> {matches[0]}")
    return matches[0]


day5_tseries_dir = find_day5_session(ANIMAL_DIR, day_number=5)

## Load `norm_spatial_activity` (preproc.h5) + `analysis_reliable_cells`/`valid_cells_mask` (Phase 3's saved SMI output)

In [ ]:
preproc_files = glob.glob(os.path.join(day5_tseries_dir, "*preproc*.h5"))
if not preproc_files:
    raise FileNotFoundError(f"No *preproc*.h5 found in {day5_tseries_dir}")
preproc_data = files.read_h5(preproc_files[0])
norm_spatial_activity = preproc_data['norm_spatial_activity']
combined_reliable = preproc_data['combined_reliable']  # raw reliability test, before onset filtering
print(f"norm_spatial_activity: {norm_spatial_activity.shape}")
print(f"combined_reliable: {int(combined_reliable.sum())}/{len(combined_reliable)} cells")

smi_files = glob.glob(os.path.join(day5_tseries_dir, "*_smi_results_dreadd.h5"))
if not smi_files:
    raise FileNotFoundError(f"No *_smi_results_dreadd.h5 found in {day5_tseries_dir}")
with h5py.File(smi_files[0], 'r') as f:
    analysis_reliable_cells = f['global_smi/analysis_reliable_cells'][:]
    valid_cells_mask = f['global_smi/valid_cells_mask'][:]
print(f"analysis_reliable_cells: {int(analysis_reliable_cells.sum())}/{len(analysis_reliable_cells)} cells")
print(f"valid_cells_mask:        {int(valid_cells_mask.sum())}/{len(valid_cells_mask)} cells")

# "Reliability test" AND "SMI validity" combined -- swap analysis_reliable_cells
# for combined_reliable on the line below if you meant the raw (pre-onset-filter) test instead.
reliable_and_valid = analysis_reliable_cells & valid_cells_mask
print(f"\nreliable_and_valid (both criteria): {int(reliable_and_valid.sum())}/{len(reliable_and_valid)} cells")

## Response plot

`create_response_plot`'s own filter step (`reliable_indices = np.where(reliable_cells)[0]`) works with any boolean mask, not just a literal "reliable_cells" array -- passing `reliable_and_valid` here restricts the plot to cells passing both criteria, with no changes needed to the function itself.

In [ ]:
fig, sorted_reliable_indices = create_response_plot(norm_spatial_activity, reliable_and_valid, clim=(0, 1))
fig.suptitle(f"JSY090, Day5 -- reliable & SMI-valid cells (n={int(reliable_and_valid.sum())})", fontsize=16)
plt.show()